**Scraping de estados de emergencias**

Grupo 3 - Corresponde la temporada de: 1 de enero al 31 de mayo de 2024

1. Descarguen e impriman el archivo https://www.gob.pe/robots.txt. En una celda de texto, expliquen qué prohíbe, si la página de búsquedas (/busquedas) está permitida y por qué igual conviene hacer pausas entre página y página (fíjense en el Crawl-delay que les piden a otros bots).

In [1]:
# Importa la biblioteca requests, que permite realizar solicitudes a páginas web.
import requests

# Define una cabecera que identifica nuestra solicitud como proveniente de un navegador.
headers = {"User-Agent": "Mozilla/5.0"}

# Realiza una solicitud HTTP para obtener el archivo robots.txt de gob.pe.
robots = requests.get(

    # Indica la dirección web del archivo que queremos consultar.
    "https://www.gob.pe/robots.txt",

    # Envía la cabecera que definimos anteriormente.
    headers=headers,

    # Establece un tiempo máximo de espera de 30 segundos.
    timeout=30
)

# Imprime los primeros 600 caracteres del contenido recibido.
print(robots.text[:600])

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



a) ¿Qué prohíbe?

Disallow: /admin/   : los robots no deben entrar en la sección administrativa de la página.
Disallow: /*?sheet= : los robots no deben visitar determinadas direcciones que utilizan el parámetro sheet.
AhrefsBot → Disallow: / : el sitio prohíbe que un robot específico, llamado AhrefsBot, rastree cualquiera de sus páginas.

b) Permiso de página de búsquedas

El archivo robots.txt no dice nada respecto a prohibición a rutas "/busquedas" por lo que se puede decir que está permitido.

c) ¿Por qué conviene hacer pausas entre página y página?

Porque: 

GPTBot → Crawl-delay: 5
Solicita una pausa de 5 segundos entre peticiones.
OAI-SearchBot → Crawl-delay: 2
Solicita una pausa de 2 segundos entre peticiones.
Quiere decir que el sitio limita la frecuencia de consultas. Por eso es mejor hacer las pausas para evitar que el servidor se sobrecargue. 

2. Con Selenium, abran la búsqueda de un solo mes de su temporada. Usen WebDriverWait, no solo time.sleep().

In [2]:
# Instala la biblioteca Selenium.
#%pip install selenium

In [3]:
# Crea una variable llamada url_base para guardar la dirección del buscador.
url_base = (

    # Indica la dirección principal del buscador de gob.pe.
    "https://www.gob.pe/busquedas?"

    # Limita la búsqueda a contenidos clasificados como normas.
    "contenido[]=normas&"

    # Aplica el filtro de institución: PCM.
    "institucion[]=pcm&"

    # Establece los términos que queremos buscar.
    # %20 representa un espacio dentro de la dirección web.
    "term=estado%20de%20emergencia%20precipitaciones&"

    # Ordena los resultados comenzando por los más recientes.
    "sort_by=recent"
)


# Crea una nueva URL añadiendo a la búsqueda el periodo de enero de 2024.
url_enero = url_base + "&desde=01-01-2024&hasta=31-01-2024"


# Muestra la URL completa para comprobar que se construyó correctamente.
print(url_enero)

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024


In [4]:
# Importa webdriver, que permite controlar un navegador web desde Python.
from selenium import webdriver

# Importa By, que permite indicar cómo localizar elementos en una página web.
from selenium.webdriver.common.by import By

# Importa WebDriverWait, que permite esperar hasta que se cumpla una condición.
from selenium.webdriver.support.ui import WebDriverWait

# Importa las condiciones que podemos utilizar durante la espera.
# EC es un nombre abreviado de expected_conditions.
from selenium.webdriver.support import expected_conditions as EC


# Inicia una nueva sesión de Chrome controlada por Selenium.
# Guarda el navegador en la variable "navegador".
navegador = webdriver.Chrome()


# Ordena al navegador que abra la URL de enero que construimos anteriormente.
navegador.get(url_enero)


# Establece una espera de hasta 30 segundos para que se cumpla una condición.
WebDriverWait(navegador, 30).until(

    # Define la condición: debe existir un elemento que coincida con nuestra búsqueda.
    EC.presence_of_element_located(

        # Indica que buscaremos el elemento mediante un selector CSS.
        # "article" es la etiqueta HTML que contiene cada resultado de búsqueda.
        (By.CSS_SELECTOR, "article")
    )
)


# Muestra un mensaje cuando se ha encontrado al menos un elemento "article".
print("La página de búsqueda ha cargado y se encontró un resultado.")

La página de búsqueda ha cargado y se encontró un resultado.


In [5]:
# Establece una espera máxima de 30 segundos en el navegador.
WebDriverWait(navegador, 30).until(

    # Indica que la espera terminará cuando se encuentre el elemento solicitado.
    EC.presence_of_element_located(

        # Busca un elemento HTML llamado "article" mediante un selector CSS.
        (By.CSS_SELECTOR, "article")
    )
)

<selenium.webdriver.remote.webelement.WebElement (session="ecb9c60946b443b5450464552e0384e5", element="f.8A85097AF52A980F465896B97C024598.d.C917973B93F05E55E9D8F5BA48422092.e.19")>

In [6]:
# Muestra la dirección web de la página que Chrome tiene abierta.
print("URL actual:", navegador.current_url)


# Busca todos los elementos HTML llamados "article".
# Guarda los elementos encontrados en una lista llamada "articulos".
articulos = navegador.find_elements(By.CSS_SELECTOR, "article")


# Cuenta cuántos elementos hay en la lista "articulos".
# Muestra esa cantidad en pantalla.
print("Número de artículos encontrados:", len(articulos))


# Comprueba si la lista "articulos" contiene al menos un elemento.
if articulos:

    # Imprime un salto de línea y un título para identificar el contenido.
    print("\nContenido del primer artículo:")

    # Selecciona el primer artículo, obtiene su texto y muestra sus primeros 500 caracteres.
    print(articulos[0].text[:500])

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024
Número de artículos encontrados: 11

Contenido del primer artículo:
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 010-2024-PCM
Publicado: 30 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Descargar archivo


In [7]:
# Comprueba si Python todavía puede comunicarse con Chrome.
print(navegador.current_url)

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024


In [8]:
# Busca el cuerpo de la página y guarda el elemento encontrado.
cuerpo = navegador.find_element(By.TAG_NAME, "body")

# Comprueba que Selenium encontró el elemento.
print("Se encontró el cuerpo de la página.")

Se encontró el cuerpo de la página.


In [9]:
# Muestra los primeros 1000 caracteres del texto visible de la página.
print(cuerpo.text[:1000])

Saltar a contenido principal
Inicio
El Estado
PCM
Resultado de búsquedas
Buscar en Gob.pe
Por institución:
Todas las instituciones
Por fecha de publicación:
01/01/24 - 31/01/24
Todo
Orientación, trámites y servicios
Noticias
Campañas
Documentos y publicaciones
Normativa
Información institucional
Filtros(2)
11 Resultados
Ordenar por:
Más recientes
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 010-2024-PCM
Publicado: 30 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Descargar archivo
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 009-2024-PCM
Publicado: 29 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los dis

3) Lean el número total de resultados que muestra la página ("N Resultados").

>**Objetivo:** identificar y extraer el número total de resultados que anuncia el buscador de gob.pe para enero de 2024.

>**Estrategia:** primero inspeccionaremos el texto visible de la página para localizar la expresión «N Resultados». Una vez identificado ese patrón, utilizaremos Python para extraer el número y guardarlo en la variable `total_resultados`.

>Este valor representa la cantidad anunciada por el buscador y servirá posteriormente para comprobar si hemos extraído todos los registros.


In [10]:
# Importa la biblioteca para buscar patrones dentro de un texto.
import re

# Busca un número seguido de la palabra "Resultados".
coincidencia = re.search(r"(\d+)\s+Resultados", cuerpo.text)

# Comprueba si se encontró el patrón esperado.
if coincidencia:

    # Extrae el número encontrado y lo convierte en un entero.
    total_resultados = int(coincidencia.group(1))

    # Muestra el total anunciado por la página.
    print("Total anunciado:", total_resultados)

    # Muestra cuántos artículos encontró Selenium.
    print("Artículos encontrados:", len(articulos))

    # Comprueba si ambas cantidades coinciden.
    print("¿Coinciden?", total_resultados == len(articulos))

else:

    # Informa si no se encontró el texto esperado.
    print("No se encontró el total de resultados.")

Total anunciado: 11
Artículos encontrados: 11
¿Coinciden? True


4) De cada resultado, extraigan:

número de la norma: es el texto del enlace principal (por ejemplo: Decreto Supremo N.° 044-2023-PCM);

fecha de publicación: el texto que empieza con Publicado:;

título: el párrafo debajo de la fecha;

enlace a la norma.

Pista: cada resultado es un <article>. Primero encuentren todos los article y luego busquen cada dato dentro de cada uno.

In [11]:
# Guarda el primer artículo de la búsqueda de enero.
primer_articulo = articulos[0]

# Busca todos los enlaces que están dentro del primer artículo.
enlaces = primer_articulo.find_elements(By.CSS_SELECTOR, "a[href]")

# Recorre los enlaces encontrados, uno por uno.
for enlace in enlaces:

    # Muestra el texto visible del enlace.
    print("Texto:", enlace.text)

    # Muestra la dirección web a la que conduce.
    print("URL:", enlace.get_attribute("href"))

    # Imprime una línea para separar los enlaces.
    print("---")

Texto: Decreto Supremo N.° 010-2024-PCM
URL: https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm
---
Texto: Descargar archivo
URL: https://cdn.www.gob.pe/uploads/document/file/6103394/5398152-ds-n-0010-2024-pcm.pdf
---


In [12]:
# Obtiene el texto visible del primer artículo.
texto_articulo = primer_articulo.text

# Divide el texto del artículo en líneas.
lineas = texto_articulo.splitlines()

# Recorre las líneas para identificar dónde están los datos.
for numero, linea in enumerate(lineas):

    # Muestra el número de línea y su contenido.
    print(numero, ":", linea)

0 : PCM - Presidencia del Consejo de Ministros
1 : Normas y documentos legales
2 : ›
3 : Decretos
4 : Decreto Supremo N.° 010-2024-PCM
5 : Publicado: 30 de enero de 2024
6 : Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
7 : Descargar archivo


In [13]:
# Busca el enlace principal del primer artículo.
enlace_principal = primer_articulo.find_element(
    By.CSS_SELECTOR, "a[href*='/institucion/pcm/normas-legales/']"
)

# Obtiene el número de la norma a partir del texto del enlace.
numero_norma = enlace_principal.text

# Obtiene la dirección web de la norma.
enlace_norma = enlace_principal.get_attribute("href")

# Busca la línea que comienza con "Publicado:".
linea_fecha = next(
    linea for linea in lineas if linea.startswith("Publicado:")
)

# Elimina la palabra "Publicado:" para conservar únicamente la fecha.
fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

# Obtiene el título que aparece inmediatamente después de la fecha.
titulo = lineas[lineas.index(linea_fecha) + 1]

# Muestra los cuatro datos extraídos.
print("Número:", numero_norma)
print("Fecha:", fecha_publicacion)
print("Título:", titulo)
print("Enlace:", enlace_norma)

Número: Decreto Supremo N.° 010-2024-PCM
Fecha: 30 de enero de 2024
Título: Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Enlace: https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm


In [14]:
# Crea una lista vacía donde guardaremos los decretos.
decretos_enero = []

# Recorre los artículos encontrados en la búsqueda de enero.
for articulo in articulos:

    # Busca el enlace principal de la norma dentro del artículo.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma a partir del texto del enlace.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Obtiene el texto visible del artículo y lo divide en líneas.
    lineas = articulo.text.splitlines()

    # Busca la línea que comienza con la palabra "Publicado:".
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Elimina la palabra "Publicado:" para conservar la fecha.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los cuatro datos del decreto en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el diccionario a nuestra lista de decretos.
    decretos_enero.append(decreto)

# Muestra cuántos decretos hemos guardado.
print("Decretos extraídos:", len(decretos_enero))

Decretos extraídos: 11


In [15]:
# Importa pandas para organizar los datos en una tabla.
import pandas as pd

# Convierte la lista de decretos en una tabla.
tabla_enero = pd.DataFrame(decretos_enero)

# Muestra las primeras cinco filas de la tabla.
display(tabla_enero)

# Comprueba si extrajimos la cantidad anunciada por el buscador.
print("¿Coinciden las cantidades?", len(tabla_enero) == total_resultados)

# Comprueba si hay datos faltantes en alguna columna.
print("Datos faltantes por columna:")
print(tabla_enero.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


5.Repitan los pasos 2 a 4 para todos los meses de su temporada. Hagan una pausa de al menos 2 segundos entre cada página.

FEBRERO

In [16]:
# Construye la URL de febrero utilizando la URL base del profesor.
url_febrero = url_base + "&desde=01-02-2024&hasta=29-02-2024"

# Abre en Chrome la búsqueda correspondiente a febrero.
navegador.get(url_febrero)

# Espera hasta 30 segundos a que aparezca el cuerpo de la página.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

# Muestra la dirección para comprobar que estamos en febrero.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-02-2024&hasta=29-02-2024


In [17]:
# Obtiene el texto visible de la página.
texto_febrero = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_febrero = re.search(
    r"(\d+)\s+Resultados",
    texto_febrero
)

# Comprueba si encontramos el total anunciado.
if coincidencia_febrero:

    # Convierte el número encontrado en un entero.
    total_febrero = int(coincidencia_febrero.group(1))

    # Muestra el total anunciado por el buscador.
    print("Total anunciado:", total_febrero)

else:

    # Informa si no encontramos el total.
    print("No se encontró el total anunciado.")

# Busca los artículos presentes en la página.
articulos_febrero = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra cuántos artículos encontró Selenium.
print("Artículos en la página:", len(articulos_febrero))

Total anunciado: 11
Artículos en la página: 11


Comprobar si hay un enlace como "siguiente". Buscamos saber si todo está en la misma página.

In [18]:
# Busca todos los enlaces de la página que tienen una dirección web.
enlaces_pagina = navegador.find_elements(By.CSS_SELECTOR, "a[href]")

# Recorre los enlaces encontrados.
for enlace in enlaces_pagina:

    # Obtiene el texto visible del enlace.
    texto_enlace = enlace.text.strip()

    # Obtiene la dirección web del enlace.
    direccion = enlace.get_attribute("href")

    # Comprueba si el enlace parece estar relacionado con la paginación.
    if (
        "siguiente" in texto_enlace.lower()
        or "página" in texto_enlace.lower()
        or "page=" in direccion.lower()
    ):

        # Muestra el texto del enlace.
        print("Texto:", texto_enlace)

        # Muestra su dirección web.
        print("URL:", direccion)

        # Separa visualmente los enlaces encontrados.
        print("---")

In [19]:
# Busca elementos que podrían utilizarse para navegar entre páginas.
controles = navegador.find_elements(
    By.CSS_SELECTOR,
    "a, button"
)

# Recorre los controles encontrados.
for control in controles:

    # Obtiene el texto visible del control.
    texto = control.text.strip()

    # Obtiene su descripción, si existe.
    descripcion = control.get_attribute("aria-label") or ""

    # Comprueba si el texto o la descripción mencionan la navegación.
    if any(
        palabra in (texto + " " + descripcion).lower()
        for palabra in ["siguiente", "anterior", "página", "paginación"]
    ):

        # Muestra el texto del control.
        print("Texto:", texto)

        # Muestra su descripción.
        print("Descripción:", descripcion)

        # Muestra el tipo de elemento HTML.
        print("Elemento:", control.tag_name)

        # Separa los resultados.
        print("---")

Extraemos los decretos de febrero

In [20]:
# Crea una lista vacía para guardar los decretos de febrero.
decretos_febrero = []

# Recorre los artículos encontrados en febrero.
for articulo in articulos_febrero:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Obtiene el texto del artículo y lo separa en líneas.
    lineas = articulo.text.splitlines()

    # Busca la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Elimina la palabra "Publicado:" para conservar la fecha.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título situado inmediatamente después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los cuatro datos en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Guarda el decreto en la lista de febrero.
    decretos_febrero.append(decreto)

# Muestra la cantidad de decretos extraídos.
print("Decretos extraídos en febrero:", len(decretos_febrero))

Decretos extraídos en febrero: 11


Comprobamos los resultados

In [21]:
# Convierte los decretos de febrero en una tabla.
tabla_febrero = pd.DataFrame(decretos_febrero)

# Muestra todos los registros de febrero.
display(tabla_febrero)

# Compara los registros extraídos con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_febrero) == total_febrero)

# Comprueba si alguna columna contiene valores faltantes.
print("Datos faltantes por columna:")
print(tabla_febrero.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 021-2024-PCM,28 de febrero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 020-2024-PCM,26 de febrero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
3,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
4,Decreto Supremo N.° 019-2024-PCM,13 de febrero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 018-2024-PCM,13 de febrero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Resolución N.° 0506-2024-TCE-S5,9 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
7,Decreto Supremo N.° 016-2024-PCM,9 de febrero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
8,Resolución N.° 0437-2024-TCE-S3,5 de febrero de 2024,Braun Medical Perú S.A. contra el otorgamiento...,https://www.gob.pe/institucion/oece/normas-leg...
9,Decreto Supremo N.° 013-2024-PCM,5 de febrero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


MARZO

Abrir la web para marzo

In [22]:
# Construye la URL de marzo de 2024.
url_marzo = url_base + "&desde=01-03-2024&hasta=31-03-2024"

# Abre la búsqueda de marzo en Chrome.
navegador.get(url_marzo)

# Espera hasta 30 segundos a que aparezca el cuerpo de la página.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

# Muestra la URL para comprobar que corresponde a marzo.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-03-2024&hasta=31-03-2024


Comprobar los resultados

In [23]:
# Obtiene el texto visible de la página.
texto_marzo = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_marzo = re.search(r"(\d+)\s+Resultados", texto_marzo)

# Comprueba que se haya encontrado el total.
if coincidencia_marzo:

    # Convierte el total anunciado en un número entero.
    total_marzo = int(coincidencia_marzo.group(1))

    # Muestra el total anunciado por el buscador.
    print("Total anunciado:", total_marzo)

else:

    # Informa si no se encontró el total.
    print("No se encontró el total anunciado.")

# Busca los artículos cargados en la página.
articulos_marzo = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra cuántos artículos encontró Selenium.
print("Artículos encontrados:", len(articulos_marzo))

Total anunciado: 11
Artículos encontrados: 11


Extracción de decretos de marzo

In [24]:
# Crea una lista vacía para guardar los decretos de marzo.
decretos_marzo = []

# Recorre los artículos encontrados en la búsqueda de marzo.
for articulo in articulos_marzo:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Obtiene el texto del artículo y lo divide en líneas.
    lineas = articulo.text.splitlines()

    # Busca la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Elimina la palabra "Publicado:" para conservar la fecha.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los cuatro datos en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el decreto a la lista de marzo.
    decretos_marzo.append(decreto)

# Convierte la lista de decretos en una tabla.
tabla_marzo = pd.DataFrame(decretos_marzo)

# Muestra los registros extraídos.
display(tabla_marzo)

# Comprueba si la cantidad extraída coincide con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_marzo) == total_marzo)

# Comprueba si existen valores faltantes.
print("Datos faltantes por columna:")
print(tabla_marzo.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 034-2024-PCM,27 de marzo de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 033-2024-PCM,26 de marzo de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 032-2024-PCM,26 de marzo de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 031-2024-PCM,26 de marzo de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Resolución N.° 0991-2024-TCE-S6,21 de marzo de 2024,Recurso de apelación interpuesto por el postor...,https://www.gob.pe/institucion/oece/normas-leg...
5,Resolución N.° 0972-2024-TCE-S2,20 de marzo de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
6,Decreto Supremo N.° 030-2024-PCM,19 de marzo de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Decreto Supremo N.° 027-2024-PCM,14 de marzo de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
8,Decreto Supremo N.° 026-2024-PCM,7 de marzo de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 025-2024-PCM,4 de marzo de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


ABRIL

Buscar abril

In [25]:
# Importa la biblioteca que permite introducir pausas.
import time

# Construye la URL de abril de 2024.
url_abril = url_base + "&desde=01-04-2024&hasta=30-04-2024"

# Espera dos segundos antes de realizar otra consulta.
time.sleep(2)

# Abre la búsqueda de abril.
navegador.get(url_abril)

# Espera hasta que aparezcan los artículos de los resultados.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
)

# Muestra la dirección de la página abierta.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-04-2024&hasta=30-04-2024


In [26]:
# Obtiene el texto visible de la página.
texto_abril = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_abril = re.search(r"(\d+)\s+Resultados", texto_abril)

# Obtiene el total anunciado por el buscador.
total_abril = int(coincidencia_abril.group(1))

# Encuentra los artículos cargados en la página.
articulos_abril = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra el total anunciado.
print("Total anunciado:", total_abril)

# Muestra la cantidad de artículos encontrados.
print("Artículos encontrados:", len(articulos_abril))

Total anunciado: 15
Artículos encontrados: 15


Extraer los decretos de abril

In [27]:
# Crea una lista vacía para guardar los decretos de abril.
decretos_abril = []

# Recorre los artículos encontrados en abril.
for articulo in articulos_abril:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Divide el texto del artículo en líneas.
    lineas = articulo.text.splitlines()

    # Identifica la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Extrae la fecha de publicación.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los datos del decreto en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el decreto a la lista de abril.
    decretos_abril.append(decreto)

# Convierte los decretos de abril en una tabla.
tabla_abril = pd.DataFrame(decretos_abril)

# Muestra los registros extraídos.
display(tabla_abril)

# Comprueba si la cantidad extraída coincide con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_abril) == total_abril)

# Comprueba si existen valores faltantes.
print("Datos faltantes por columna:")
print(tabla_abril.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 049-2024-PCM,30 de abril de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 048-2024-PCM,30 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 047-2024-PCM,30 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 046-2024-PCM,30 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 045-2024-PCM,25 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 044-2024-PCM,25 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 043-2024-PCM,15 de abril de 2024,Declaratoria de Estado de Emergencia Declarar ...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Decreto Supremo N.° 042-2024-PCM,12 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
8,Decreto Supremo N.° 041-2024-PCM,12 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 040-2024-PCM,11 de abril de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


MAYO

Buscar mayo

In [28]:
# Construye la URL de mayo de 2024.
url_mayo = url_base + "&desde=01-05-2024&hasta=31-05-2024"

# Espera dos segundos antes de realizar otra consulta.
time.sleep(2)

# Abre la búsqueda de mayo.
navegador.get(url_mayo)

# Espera hasta que aparezcan los artículos de los resultados.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
)

# Muestra la dirección de la página abierta.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-05-2024&hasta=31-05-2024


Comprobar los resultados

In [29]:
# Obtiene el texto visible de la página.
texto_mayo = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_mayo = re.search(r"(\d+)\s+Resultados", texto_mayo)

# Obtiene el total anunciado por el buscador.
total_mayo = int(coincidencia_mayo.group(1))

# Encuentra los artículos cargados en la página.
articulos_mayo = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra el total anunciado.
print("Total anunciado:", total_mayo)

# Muestra la cantidad de artículos encontrados.
print("Artículos encontrados:", len(articulos_mayo))

Total anunciado: 3
Artículos encontrados: 3


Extraer los decretos de mayo

In [30]:
# Crea una lista vacía para guardar los decretos de mayo.
decretos_mayo = []

# Recorre los artículos encontrados en mayo.
for articulo in articulos_mayo:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Divide el texto del artículo en líneas.
    lineas = articulo.text.splitlines()

    # Identifica la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Extrae la fecha de publicación.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los datos del decreto en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el decreto a la lista de mayo.
    decretos_mayo.append(decreto)

# Convierte los decretos de mayo en una tabla.
tabla_mayo = pd.DataFrame(decretos_mayo)

# Muestra todos los registros extraídos.
display(tabla_mayo)

# Comprueba si la cantidad extraída coincide con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_mayo) == total_mayo)

# Comprueba si existen valores faltantes.
print("Datos faltantes por columna:")
print(tabla_mayo.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 054-2024-PCM,29 de mayo de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 052-2024-PCM,22 de mayo de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 050-2024-PCM,8 de mayo de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


6.Verificación: impriman una tabla con mes, resultados_totales y resultados_extraidos. Los dos números deben coincidir en todos los meses. Si en alguno no coinciden, revisen su código antes de seguir.

Compararemos dos objetos que ya tenemos en Python:
(i) El resultado de la extracción primaria: específicamente, el número de resultados que leímos del texto de gob.pe y guardamos en variables como total_abril.
(ii) Nuestras tablas estructuradas: los registros que extrajimos y organizamos en tablas como tabla_abril.

En teoría debería ser así: 

mes	resultados_totales	resultados_extraidos
Enero	11	11
Febrero	11	11
Marzo	11	11
Abril	15	15
Mayo	3	3

In [31]:
# Crea una lista con los resultados anunciados y extraídos de cada mes.
verificacion = [
    {"mes": "Enero", "resultados_totales": total_resultados, "resultados_extraidos": len(tabla_enero)},
    {"mes": "Febrero", "resultados_totales": total_febrero, "resultados_extraidos": len(tabla_febrero)},
    {"mes": "Marzo", "resultados_totales": total_marzo, "resultados_extraidos": len(tabla_marzo)},
    {"mes": "Abril", "resultados_totales": total_abril, "resultados_extraidos": len(tabla_abril)},
    {"mes": "Mayo", "resultados_totales": total_mayo, "resultados_extraidos": len(tabla_mayo)}
]

# Convierte la lista en una tabla de pandas.
tabla_verificacion = pd.DataFrame(verificacion)

# Muestra la tabla completa.
display(tabla_verificacion)

,mes,resultados_totales,resultados_extraidos
0,Enero,11,11
1,Febrero,11,11
2,Marzo,11,11
3,Abril,15,15
4,Mayo,3,3


Comprobamos que las cantidades coincidan

In [32]:
# Compara las dos cantidades de cada mes.
coincidencias = tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]

# Comprueba si todas las comparaciones son verdaderas.
print("¿Coinciden todos los meses?", coincidencias.all())

¿Coinciden todos los meses? True


7. Eliminen las normas repetidas.

Identificamos si hay normas repetidas. Primero unimos las tablas de todos los 5 meses y luego vamos a revisar por la clave (código de la normativa), si es que existen duplicados.

Integro las 5 tablas en una sola.

In [33]:
# Reúne las cinco tablas mensuales en una sola tabla.
tabla_decretos = pd.concat(
    [tabla_enero, tabla_febrero, tabla_marzo, tabla_abril, tabla_mayo],
    ignore_index=True
)

# Muestra la cantidad de registros reunidos.
print("Registros consolidados:", len(tabla_decretos))

Registros consolidados: 51


Identificamos si existen normas repetidas.

In [34]:
# Identifica todas las filas cuyo número de norma aparece más de una vez.
duplicados = tabla_decretos[
    tabla_decretos.duplicated(subset=["numero"], keep=False)
]

# Muestra cuántos registros tienen un número de norma repetido.
print("Registros con número repetido:", len(duplicados))

# Muestra los registros repetidos, ordenados por número de norma.
display(duplicados.sort_values("numero"))

Registros con número repetido: 0


,numero,fecha,titulo,enlace


8. Ojo: aunque la búsqueda filtra por PCM, se cuelan normas de otras instituciones (por ejemplo, resoluciones del OECE). Quédense solo con las que tienen /institucion/pcm/ en el enlace e impriman cuántas quitaron.

Nuestro criterio será revisar el enlace web. Si vemos que una normativa no fue publicada en "/institucion/pcm/", la excluiremos.

In [35]:
# Identifica los enlaces que no contienen la ruta institucional de la PCM.
no_pcm = ~tabla_decretos["enlace"].str.contains("/institucion/pcm/", na=False)

# Selecciona los registros que no pertenecen a la PCM.
tabla_excluidos = tabla_decretos[no_pcm]

# Muestra cuántos registros no corresponden a la PCM.
print("Registros que no pertenecen a la PCM:", len(tabla_excluidos))

# Muestra los registros que tendríamos que excluir.
display(tabla_excluidos)

Registros que no pertenecen a la PCM: 8


,numero,fecha,titulo,enlace
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
13,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
14,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
17,Resolución N.° 0506-2024-TCE-S5,9 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
19,Resolución N.° 0437-2024-TCE-S3,5 de febrero de 2024,Braun Medical Perú S.A. contra el otorgamiento...,https://www.gob.pe/institucion/oece/normas-leg...
26,Resolución N.° 0991-2024-TCE-S6,21 de marzo de 2024,Recurso de apelación interpuesto por el postor...,https://www.gob.pe/institucion/oece/normas-leg...
27,Resolución N.° 0972-2024-TCE-S2,20 de marzo de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
44,Resolución N.° 1071-2024-TCE-S4,1 de abril de 2024,Procedimiento administrativo sancionador gener...,https://www.gob.pe/institucion/oece/normas-leg...


Ahora que hemos identificado las normas que no pertenecen a la PCM, crearemos una nueva tabla que contenga exclusivamente los enlaces de la PCM. 

In [36]:
# Crea una tabla nueva con los registros que pertenecen a la PCM.
tabla_pcm = tabla_decretos[~no_pcm].copy()

# Crea una tabla independiente con los registros excluidos.
tabla_excluidos = tabla_decretos[no_pcm].copy()

# Muestra la cantidad de registros de cada tabla.
print("Tabla original:", len(tabla_decretos))
print("Tabla PCM:", len(tabla_pcm))
print("Tabla de excluidos:", len(tabla_excluidos))

# Comprueba que no se haya perdido ningún registro.
print("¿Coincide el total?", len(tabla_pcm) + len(tabla_excluidos) == len(tabla_decretos))

Tabla original: 51
Tabla PCM: 43
Tabla de excluidos: 8
¿Coincide el total? True


Y ahora mostramos las incluidas en la tabla PCM y las excluidas.

In [37]:
# Muestra la tabla que contiene únicamente las normas de la PCM.
display(tabla_pcm)

# Muestra las normas excluidas para poder reportarlas.
display(tabla_excluidos)

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
10,Decreto Supremo N.° 001-2024-PCM,4 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


,numero,fecha,titulo,enlace
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
13,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
14,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
17,Resolución N.° 0506-2024-TCE-S5,9 de febrero de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
19,Resolución N.° 0437-2024-TCE-S3,5 de febrero de 2024,Braun Medical Perú S.A. contra el otorgamiento...,https://www.gob.pe/institucion/oece/normas-leg...
26,Resolución N.° 0991-2024-TCE-S6,21 de marzo de 2024,Recurso de apelación interpuesto por el postor...,https://www.gob.pe/institucion/oece/normas-leg...
27,Resolución N.° 0972-2024-TCE-S2,20 de marzo de 2024,Recurso de apelación interpuesto por la empres...,https://www.gob.pe/institucion/oece/normas-leg...
44,Resolución N.° 1071-2024-TCE-S4,1 de abril de 2024,Procedimiento administrativo sancionador gener...,https://www.gob.pe/institucion/oece/normas-leg...
